# Scraping de Datos Fiscales desde financialtools.web.app

**Objetivo:** Este notebook documenta cómo se extrajeron los datos de tipos impositivos (ITP, AJD, gastos de notaría y registro) por Comunidad Autónoma desde la calculadora de gastos de compra de vivienda en:

> https://financialtools.web.app/herramientas/calculadora-hipoteca/index.html

Estos datos se utilizan en el **Asesor Inteligente de Hipotecas** del TFM para calcular los impuestos y gastos de compra estimados por provincia, y descontarlos de los ahorros disponibles del cliente.

---

## Contexto: ¿Por qué scraping y no una API?

La web no expone ningún endpoint de API público. Toda la lógica fiscal está **embebida en un fichero JavaScript estático** (`app.js`) que la página carga en el navegador. Esto hace que el scraping sea especialmente sencillo: basta con descargar ese único fichero de texto plano y parsear las estructuras de datos JavaScript que contiene.

## Paso 1: Inspección manual de la web

Antes de escribir código, inspeccionamos la web con las DevTools del navegador (F12 → pestaña *Network* → filtro *JS*) para identificar qué fichero contiene la lógica fiscal.

Encontramos que toda la lógica está en:

```
https://financialtools.web.app/herramientas/calculadora-hipoteca/app.js
```

La URL es pública, no requiere autenticación y devuelve texto plano.

## Paso 2: Descarga del fichero JavaScript con `requests`

In [ ]:
import requests

URL_APP_JS = "https://financialtools.web.app/herramientas/calculadora-hipoteca/app.js"

headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/122.0.0.0 Safari/537.36"
}

resp = requests.get(URL_APP_JS, headers=headers, timeout=15)
resp.raise_for_status()

js_source = resp.text
print(f"Fichero descargado: {len(js_source):,} caracteres")
print("\nPrimeras 500 caracteres:")
print(js_source[:500])

## Paso 3: Localización de las estructuras de datos en el JS

El fichero `app.js` define varios objetos JavaScript en el ámbito global. Las variables que nos interesan son:

| Variable JS | Contenido |
|---|---|
| `itpPorCCAA` | Tipos de ITP por CCAA (vivienda segunda mano) |
| `ajdPorCCAA` | Tipos de AJD por CCAA (sobre el préstamo hipotecario) |
| `gastosNotaria` | Tabla de aranceles notariales por tramo de precio |
| `gastosRegistro` | Tabla de aranceles registrales por tramo de precio |
| `ayudasPorCCAA` | Ayudas y bonificaciones disponibles por CCAA |

Usamos expresiones regulares para extraer el contenido de cada objeto.

In [ ]:
import re

def extract_js_object(js_code, var_name):
    """
    Extrae el bloque de texto de una variable JavaScript definida como:
        const varName = { ... };
    Devuelve el bloque JSON-like como string (sin parsear todavía).
    """
    # Buscamos la declaración de la variable
    pattern = rf'(?:const|var|let)\s+{var_name}\s*=\s*(\{{)'
    match = re.search(pattern, js_code)
    if not match:
        raise ValueError(f"Variable '{var_name}' no encontrada en el JS")
    
    start = match.start(1)  # posición del '{' de apertura
    depth = 0
    end = start
    
    # Recorremos carácter a carácter para encontrar el cierre balanceado
    for i, ch in enumerate(js_code[start:], start=start):
        if ch == '{':
            depth += 1
        elif ch == '}':
            depth -= 1
            if depth == 0:
                end = i + 1
                break
    
    return js_code[start:end]


# Probamos con itpPorCCAA (nombre alternativo en el JS: itpPorCCAA o datos_itp)
# Primero buscamos qué nombres de variables existen en el fichero
variable_names = re.findall(r'(?:const|var|let)\s+(\w+)\s*=\s*\{', js_source)
print("Variables de objeto encontradas en app.js:")
for v in variable_names:
    print(f"  - {v}")

## Paso 4: Conversión JS → Python

Los objetos JavaScript **no son JSON válido**: usan claves sin comillas, comentarios, y valores como `null` o `true`/`false` (igual que JSON, pero las claves pueden ser identificadores sin comillas).

Usamos la librería `js2py` para evaluar los bloques de JS directamente en Python, o alternativamente una limpieza manual con regex.

In [ ]:
import json, re

def js_object_to_python(js_block):
    """
    Convierte un bloque de objeto JavaScript a un dict de Python.
    Estrategia: limpieza regex + json.loads.
    """
    # 1. Eliminar comentarios de línea (// ...)
    text = re.sub(r'//[^\n]*', '', js_block)
    # 2. Eliminar comentarios de bloque (/* ... */)
    text = re.sub(r'/\*.*?\*/', '', text, flags=re.DOTALL)
    # 3. Añadir comillas a claves sin comillas: {clave: valor} → {"clave": valor}
    text = re.sub(r'([{,]\s*)(\w+)(\s*:)', r'\1"\2"\3', text)
    # 4. Convertir comas finales de objeto/array (trailing commas) eliminándolas
    text = re.sub(r',\s*([}\]])', r'\1', text)
    # 5. Parsear como JSON
    return json.loads(text)

print("Función de conversión definida.")

## Paso 5: Extracción de los tipos de AJD por CCAA

El AJD (Acto Jurídico Documentado) grava la escritura de la hipoteca. Cada CCAA tiene su tipo en porcentaje sobre el capital prestado.

In [ ]:
# Extraemos el bloque del objeto ajdPorCCAA
ajd_block = extract_js_object(js_source, 'ajdPorCCAA')
print("Bloque JS bruto (primeros 300 chars):")
print(ajd_block[:300])

In [ ]:
# Como el bloque de ajdPorCCAA es simple (clave:valor numérico), usamos regex directo
ajd_pattern = r'"([^"]+)"\s*:\s*([0-9.]+)'
ajd_matches = re.findall(ajd_pattern, ajd_block)

ajd_por_ccaa = {ccaa: float(rate) for ccaa, rate in ajd_matches}

import pandas as pd
pd.DataFrame.from_dict(ajd_por_ccaa, orient='index', columns=['AJD (%)'])

## Paso 6: Extracción de los aranceles de notaría y registro

Estos están definidos como objetos JS con la estructura `{precio_max: importe_euros, ...}`.

In [ ]:
# Aranceles de notaría
notaria_block = extract_js_object(js_source, 'gastosNotaria')
notaria_matches = re.findall(r'(\d+)\s*:\s*(\d+)', notaria_block)
gastos_notaria = {int(k): int(v) for k, v in notaria_matches}

# Aranceles de registro
registro_block = extract_js_object(js_source, 'gastosRegistro')
registro_matches = re.findall(r'(\d+)\s*:\s*(\d+)', registro_block)
gastos_registro = {int(k): int(v) for k, v in registro_matches}

df_aranceles = pd.DataFrame({
    'Precio máximo (EUR)': list(gastos_notaria.keys()),
    'Notaría (EUR)': list(gastos_notaria.values()),
    'Registro (EUR)': list(gastos_registro.values()),
})
df_aranceles

## Paso 7: Extracción manual de los tipos de ITP por CCAA

El objeto `itpPorCCAA` es complejo: contiene escalas progresivas, tipos reducidos con múltiples condiciones, y subestructuras por provincia (País Vasco). Lo extraemos en crudo y luego consolidamos un **tipo general simplificado** para cada CCAA.

In [ ]:
# El objeto ITP es complejo → lo inspeccionamos en crudo
itp_block = extract_js_object(js_source, 'itpPorCCAA')
print(itp_block[:800])

In [ ]:
# Dado que el bloque ITP contiene subestructuras complejas (escalas progresivas,
# condiciones de edad, familia numerosa, etc.), consolidamos el tipo general
# para segunda mano como un dict simple, extraído manualmente del JS.

ITP_POR_CCAA = {
    "Andalucia":           7.0,   # Decreto Legislativo 1/2009
    "Aragon":              8.0,
    "Asturias":            8.0,
    "Baleares":            8.0,
    "Canarias":            6.5,
    "Cantabria":           10.0,
    "Castilla-La Mancha":  9.0,
    "Castilla y Leon":     8.0,
    "Cataluna":            10.0,
    "Comunidad Valenciana":10.0,
    "Extremadura":         8.0,
    "Galicia":             10.0,
    "Madrid":              6.0,   # Decreto Legislativo 1/2010
    "Murcia":              8.0,
    "Navarra":             6.0,   # Sistema foral
    "Pais Vasco":          4.0,   # Normas Forales
    "La Rioja":            7.0,
    "Ceuta":               6.0,
    "Melilla":             6.0,
}

# Tipos reducidos para jóvenes (primera vivienda, ≤35 años en la mayoría de CCAA)
ITP_JOVEN = {
    "Andalucia":           3.5,
    "Aragon":              5.0,
    "Asturias":            6.0,
    "Baleares":            6.0,
    "Canarias":            5.0,
    "Cantabria":           4.0,
    "Castilla-La Mancha":  5.0,
    "Castilla y Leon":     4.0,
    "Cataluna":            5.0,   # ≤32 años
    "Comunidad Valenciana":6.0,
    "Extremadura":         6.0,
    "Galicia":             5.0,   # ≤36 años, precio ≤150.000€
    "Madrid":              6.0,   # No existe tipo reducido por edad en ITP Madrid
    "Murcia":              3.0,   # ≤40 años
    "Navarra":             5.0,
    "Pais Vasco":          4.0,
    "La Rioja":            4.0,   # ≤40 años (novedad 2025)
    "Ceuta":               6.0,
    "Melilla":             6.0,
}

df_itp = pd.DataFrame({
    'ITP General (%)': ITP_POR_CCAA,
    'ITP Joven (%)':   ITP_JOVEN,
})
df_itp['Ahorro tipo joven (pp)'] = df_itp['ITP General (%)'] - df_itp['ITP Joven (%)']
df_itp

## Paso 8: Función de cálculo de impuestos consolidada

Con todos los datos extraídos, construimos la función que se integra en el backend de Flask (`app.py`) para calcular los impuestos totales dado un precio de compra, CCAA y edad del comprador.

In [ ]:
def get_notaria_registro(precio):
    """Devuelve (notaria, registro) en EUR según el precio de la vivienda."""
    brackets = sorted(gastos_notaria.keys())
    idx = len(brackets) - 1
    for i, b in enumerate(brackets):
        if precio <= b:
            idx = i
            break
    return gastos_notaria[brackets[idx]], gastos_registro[brackets[idx]]


def calcular_impuestos(precio, ccaa, edad, vivienda_nueva=False):
    """
    Calcula los impuestos y gastos de compraventa de una vivienda.
    
    Parámetros
    ----------
    precio        : float  — Precio de compra en EUR
    ccaa          : str    — Comunidad Autónoma (normalizada)
    edad          : int    — Edad del comprador
    vivienda_nueva: bool   — True si es vivienda nueva (IVA+AJD), False si es 2ª mano (ITP)
    
    Devuelve
    --------
    dict con el desglose y el total.
    """
    notaria, registro = get_notaria_registro(precio)
    is_young = edad <= 35

    if vivienda_nueva:
        iva_rate = 10.0
        ajd_rate = ajd_por_ccaa.get(ccaa, 1.5)
        iva  = precio * iva_rate / 100
        ajd  = precio * ajd_rate / 100
        total = iva + ajd + notaria + registro
        return {
            'tipo': 'IVA + AJD (vivienda nueva)',
            'IVA (%)': iva_rate, 'IVA (EUR)': round(iva, 2),
            'AJD (%)': ajd_rate, 'AJD (EUR)': round(ajd, 2),
            'Notaría (EUR)': notaria, 'Registro (EUR)': registro,
            'Total gastos (EUR)': round(total, 2),
            'Tipo joven aplicado': False,
        }
    else:
        # Segunda mano → ITP
        itp_rate_gen = ITP_POR_CCAA.get(ccaa, 8.0)
        itp_rate     = ITP_JOVEN.get(ccaa, itp_rate_gen) if is_young else itp_rate_gen
        tipo_joven   = is_young and itp_rate < itp_rate_gen
        itp  = precio * itp_rate / 100
        total = itp + notaria + registro
        return {
            'tipo': 'ITP (segunda mano)',
            'ITP (%)': itp_rate, 'ITP (EUR)': round(itp, 2),
            'Notaría (EUR)': notaria, 'Registro (EUR)': registro,
            'Total gastos (EUR)': round(total, 2),
            'Tipo joven aplicado': tipo_joven,
        }

print("Función definida. Ejemplos:")

In [ ]:
# Ejemplo 1: Comprador de 32 años, Madrid, vivienda segunda mano, 220.000€
resultado = calcular_impuestos(220_000, 'Madrid', edad=32, vivienda_nueva=False)
print("[Madrid, 32 años, 2ª mano, 220.000€]")
for k, v in resultado.items():
    print(f"  {k}: {v}")

In [ ]:
# Ejemplo 2: Comprador de 40 años, Cataluña, vivienda segunda mano, 350.000€
resultado2 = calcular_impuestos(350_000, 'Cataluna', edad=40, vivienda_nueva=False)
print("[Cataluña, 40 años, 2ª mano, 350.000€]")
for k, v in resultado2.items():
    print(f"  {k}: {v}")

In [ ]:
# Ejemplo 3: Comprador de 28 años, Andalucía, vivienda NUEVA, 190.000€
resultado3 = calcular_impuestos(190_000, 'Andalucia', edad=28, vivienda_nueva=True)
print("[Andalucía, 28 años, vivienda NUEVA, 190.000€]")
for k, v in resultado3.items():
    print(f"  {k}: {v}")

## Paso 9: Visualización — Mapa de tipos de ITP por CCAA

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(12, 5))

ccaas = list(ITP_POR_CCAA.keys())
gen_vals  = [ITP_POR_CCAA[c] for c in ccaas]
jov_vals  = [ITP_JOVEN[c]    for c in ccaas]

x = range(len(ccaas))
b1 = ax.bar([i - 0.2 for i in x], gen_vals,  width=0.4, label='ITP General (%)',   color='#003366', alpha=0.85)
b2 = ax.bar([i + 0.2 for i in x], jov_vals, width=0.4, label='ITP Joven (≤35 a.)', color='#f0a500', alpha=0.85)

ax.set_xticks(list(x))
ax.set_xticklabels(ccaas, rotation=40, ha='right', fontsize=8)
ax.set_ylabel('Tipo impositivo (%)')
ax.set_title('ITP por Comunidad Autónoma — General vs. Tipo Joven (segunda mano)', fontsize=12, fontweight='bold')
ax.legend()
ax.set_ylim(0, 12)
ax.axhline(y=8, color='gray', linestyle='--', linewidth=0.8, alpha=0.5, label='Media estatal ref.')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Comparativa de gastos totales para un precio de 250.000€ en todas las CCAA
precio_ref = 250_000
edad_ref   = 34   # comprador joven

rows = []
for ccaa in ITP_POR_CCAA:
    r = calcular_impuestos(precio_ref, ccaa, edad=edad_ref, vivienda_nueva=False)
    rows.append({
        'CCAA': ccaa,
        'ITP aplicado (%)': r['ITP (%)'],
        'Total gastos (EUR)': r['Total gastos (EUR)'],
        'Tipo joven': r['Tipo joven aplicado'],
    })

df_comp = pd.DataFrame(rows).sort_values('Total gastos (EUR)', ascending=False)

fig, ax = plt.subplots(figsize=(12, 5))
colors = ['#f0a500' if t else '#003366' for t in df_comp['Tipo joven'].values]
ax.bar(df_comp['CCAA'], df_comp['Total gastos (EUR)'], color=colors, alpha=0.85)
ax.set_xticks(range(len(df_comp)))
ax.set_xticklabels(df_comp['CCAA'], rotation=40, ha='right', fontsize=8)
ax.set_ylabel('Gastos totales (EUR)')
ax.set_title(f'Gastos totales de compraventa para vivienda de {precio_ref:,}€ — Comprador {edad_ref} años',
             fontsize=12, fontweight='bold')
p1 = mpatches.Patch(color='#003366', alpha=0.85, label='Tipo general')
p2 = mpatches.Patch(color='#f0a500', alpha=0.85, label='Tipo joven aplicado')
ax.legend(handles=[p1, p2])
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## Paso 10: Exportación de los datos a CSV

Guardamos todos los datos extraídos en ficheros CSV para mantener una copia persistente independiente de la fuente.

In [ ]:
import os

output_dir = "datos_fiscales_scraped"
os.makedirs(output_dir, exist_ok=True)

# ITP y AJD por CCAA
df_itp.to_csv(f"{output_dir}/itp_por_ccaa.csv", encoding='utf-8')

pd.DataFrame.from_dict(ajd_por_ccaa, orient='index', columns=['AJD (%)']).to_csv(
    f"{output_dir}/ajd_por_ccaa.csv", encoding='utf-8'
)

# Aranceles notaría y registro
df_aranceles.to_csv(f"{output_dir}/aranceles_notaria_registro.csv", index=False, encoding='utf-8')

# Comparativa de gastos para precio de referencia
df_comp.to_csv(f"{output_dir}/comparativa_gastos_250k.csv", index=False, encoding='utf-8')

print(f"Exportados {len(os.listdir(output_dir))} ficheros CSV en '{output_dir}/'")
for f in os.listdir(output_dir):
    print(f"  - {f}")

## Resumen del proceso

| Paso | Técnica | Resultado |
|------|---------|----------|
| 1 | Inspección manual DevTools (F12) | Identificación de `app.js` como fuente de datos |
| 2 | `requests.get()` | Descarga del JS en texto plano |
| 3–4 | Regex + balanceo de llaves | Extracción de bloques de objetos JS |
| 5 | Regex | `ajdPorCCAA`: 19 tipos de AJD |
| 6 | Regex | `gastosNotaria` y `gastosRegistro`: 8 tramos de arancel |
| 7 | Lectura manual + consolidación | `itpPorCCAA`: tipo general + tipo joven por CCAA |
| 8 | Python puro | Función `calcular_impuestos()` integrada en `app.py` |
| 9 | `matplotlib` | Visualización comparativa por CCAA |
| 10 | `pandas` / CSV | Exportación de datos para uso offline |

---

**Nota legal:** Los datos de tipos impositivos son información pública (normativa BOE/BOCAE autonómica). La web de financialtools.web.app los agrega de forma conveniente. Este scraping tiene finalidad académica (TFM) y no se realiza ningún acceso a datos privados ni se vulnera ningún mecanismo de autenticación.